<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/app/01_prepare_app_data_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# =========================
# BLOCK 1 — Prepare artefacts for AppDemo
# =========================

import os, re, glob, shutil, datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Config
IN_COLAB = (os.environ.get("COLAB_RELEASE_TAG") is not None) or ("/content" in str(Path.cwd()))
from google.colab import drive  # safe in Colab; ignored elsewhere
if IN_COLAB:
    try:
        drive.mount("/content/drive", force_remount=False)
        print("Mounted at /content/drive")
    except Exception:
        print("Drive already mounted at /content/drive")

# Input CSV (engineered version with 'Date' column)
INPUT_CSV = "/content/drive/MyDrive/gt-markets/data/processed/merged_financial_trends_engineered_2025-09-07.csv"
print(f"Input: {INPUT_CSV}")

# Output roots
DRIVE_ARTEFACTS = Path("/content/drive/MyDrive/gt-markets/AppDemo/artefacts")
REPO_ARTEFACTS  = Path("/content/gt-markets/AppDemo/artefacts")
print(f"Drive artefacts: {DRIVE_ARTEFACTS}")
print(f"Repo artefacts: {REPO_ARTEFACTS}")

# Create base dirs
for p in [DRIVE_ARTEFACTS, REPO_ARTEFACTS]:
    p.mkdir(parents=True, exist_ok=True)
    for a in ["BTC", "GOLD", "OIL", "USDCNY"]:
        (p / a).mkdir(parents=True, exist_ok=True)

# ---- Helpers
def pick_series(df: pd.DataFrame, ticker: str, field: str = "Close") -> pd.Series:
    """Pick a column like 'BTC-USD Close' or 'GC=F Close'."""
    candidates = [c for c in df.columns if c.endswith(f" {field}") and c.startswith(ticker)]
    if not candidates:
        raise KeyError(f"Missing series for {ticker} {field}")
    s = df[candidates[0]].astype(float)
    s.name = f"{ticker} {field}"
    return s

def daily_to_weekly_close(s: pd.Series) -> pd.Series:
    """Resample to W-FRI (last close of the week)."""
    return s.resample("W-FRI").last().dropna()

def compute_returns(close: pd.Series) -> pd.Series:
    """Simple daily/weekly pct change; fill initial NA with 0."""
    ret = close.pct_change(fill_method=None).fillna(0.0)
    ret.name = "ret"
    return ret

def run_baseline_metrics(close: pd.Series) -> pd.DataFrame:
    """Build BASE_SMA and BASE_EMA strategies and summarise."""
    ret = compute_returns(close)

    sma = close.rolling(20, min_periods=1).mean()
    ema = close.ewm(span=20, adjust=False).mean()

    sig_sma = (close > sma).astype(int).rename("signal")
    sig_ema = (close > ema).astype(int).rename("signal")

    def backtest(signal: pd.Series, name: str) -> dict:
        # Trades on next period’s return using today’s signal
        strat_ret = ret.shift(-1).fillna(0.0) * signal
        total = (1.0 + strat_ret).prod() - 1.0
        trades = int(signal.diff().abs().fillna(0).sum())
        hit = float((strat_ret > 0).mean())
        return dict(strategy=name, total_return=total, trades=trades, hit_rate=hit)

    out = pd.DataFrame([
        backtest(sig_sma, "BASE_SMA"),
        backtest(sig_ema, "BASE_EMA"),
    ])
    out["type"] = "baseline"
    return out[["type", "strategy", "total_return", "trades", "hit_rate"]]

def kw_signal_from_file(fp: Path) -> pd.Series:
    """Load ‘signals_KW_*.csv’ and return an aligned 0/1 signal series by Date."""
    df = pd.read_csv(fp)
    # Date column normalisation
    date_col = "Date" if "Date" in df.columns else ("date" if "date" in df.columns else None)
    if not date_col:
        raise KeyError(f"{fp.name}: no Date/date column")
    idx = pd.to_datetime(df[date_col], errors="coerce")
    df = df.set_index(idx).sort_index()
    # Find a signal-like column
    cand = None
    for c in ["signal", "pred", "proba", "y_hat", "score", "prediction"]:
        if c in df.columns:
            cand = c
            break
    if cand is None:
        # fall back to any numeric column
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if not num_cols:
            raise KeyError(f"{fp.name}: no numeric columns to infer signal")
        cand = num_cols[0]
    raw = pd.to_numeric(df[cand], errors="coerce")

    # Threshold: if looks like probability use >=0.5; else >0 means long
    if raw.dropna().between(0, 1).all():
        sig = (raw >= 0.5).astype(int)
    else:
        sig = (raw > 0).astype(int)

    sig = sig.rename("signal").astype("Int64")  # allow NaN during alignment
    return sig

def kw_metrics_for_model(close: pd.Series, sig: pd.Series, strategy_name: str) -> dict:
    """Apply the given discrete signal to the close series and summarise."""
    ret = compute_returns(close)
    # Align to price index; gaps treated as no-position (0)
    aligned = sig.reindex(close.index).fillna(0).astype(int)
    strat_ret = ret.shift(-1).fillna(0.0) * aligned
    total = (1.0 + strat_ret).prod() - 1.0
    trades = int(aligned.diff().abs().fillna(0).sum())
    hit = float((strat_ret > 0).mean())
    return dict(type="keyword", strategy=strategy_name, total_return=total, trades=trades, hit_rate=hit)

def write_csv(df: pd.DataFrame, path: Path, index=False):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index)

# ---- Load merged data
df = pd.read_csv(INPUT_CSV)
# Normalise Date index
date_col = "Date" if "Date" in df.columns else ("date" if "date" in df.columns else None)
if not date_col:
    raise KeyError("Input CSV must contain 'Date' or 'date' column.")
df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df = df.set_index(date_col).sort_index()
print(f"Merged shape: {df.shape}")
print("Sample cols:", list(df.columns[:12]))

# Map assets to tickers in the merged file
TICKER_MAP = {
    "BTC": "BTC-USD",
    "GOLD": "GC=F",
    "OIL": "CL=F",
    "USDCNY": "USDCNY=X",
}

# ---- Build baseline + leaderboards (D/W)
summary_rows = []

def do_asset(asset: str):
    ticker = TICKER_MAP[asset]
    # Daily
    closeD = pick_series(df, ticker, "Close").dropna()
    baseD = run_baseline_metrics(closeD.copy())
    write_csv(baseD, DRIVE_ARTEFACTS / asset / "metrics_baseline_D.csv")
    write_csv(baseD, DRIVE_ARTEFACTS / asset / "leaderboard_D.csv")
    print(f"[ok] {asset} D: baseline rows={len(baseD)}")
    summary_rows.append(dict(asset=asset, freq="D", rows=len(baseD)))

    # Weekly
    closeW = daily_to_weekly_close(closeD)
    baseW = run_baseline_metrics(closeW.copy())
    write_csv(baseW, DRIVE_ARTEFACTS / asset / "metrics_baseline_W.csv")
    write_csv(baseW, DRIVE_ARTEFACTS / asset / "leaderboard_W.csv")
    print(f"[ok] {asset} W: baseline rows={len(baseW)}")
    summary_rows.append(dict(asset=asset, freq="W", rows=len(baseW)))

for a in TICKER_MAP:
    do_asset(a)

summary = pd.DataFrame(summary_rows)
print(f"Summary rows: {len(summary)}")

# ---- Keyword metrics (apply trading strategy BASE_SMA/BASE_EMA to ML signals)
# We scan the Drive signal dumps and write per-asset keyword metrics + root rollups.
root_kw_D = []
root_kw_W = []

SIGNAL_GLOB = "/content/drive/MyDrive/gt-markets/AppDemo/artefacts/**/signals_KW_*_*.csv"

for sig_path in sorted(glob.glob(SIGNAL_GLOB, recursive=True)):
    fp = Path(sig_path)
    # Expect filename like: signals_KW_<MODEL>_<TYPE>_<FREQ>.csv
    m = re.match(r"signals_KW_([A-Za-z0-9]+)_w?\d*_(BASE|EXT)_([DW])\.csv", fp.name)
    # Also accept signals_KW_<MODEL>_(BASE|EXT)_([DW]).csv (no window)
    n = re.match(r"signals_KW_([A-Za-z0-9]+)_(BASE|EXT)_([DW])\.csv", fp.name) if m is None else None
    hit = m or n
    if not hit:
        # Skip non-standard filenames
        continue
    model, data_type, freq = hit.group(1), hit.group(2), hit.group(3)

    # Infer asset folder from path parts
    upper_parts = [p.upper() for p in fp.parts]
    asset = None
    for a in TICKER_MAP.keys():
        if a in upper_parts:
            asset = a
            break
    if asset is None:
        continue

    ticker = TICKER_MAP[asset]
    # Build price series for this freq
    closeD = pick_series(df, ticker, "Close").dropna()
    closeW = daily_to_weekly_close(closeD)
    close = closeD if freq == "D" else closeW

    # Load and align signal
    try:
        sig = kw_signal_from_file(fp)
    except Exception as e:
        print(f"[skip] {fp.name}: {e}")
        continue

    # Two trading strategies (same as baseline) applied to ML signal
    # Strategy name shows which trading rule was used (not the ML model).
    metrics = []
    for strat_name in ["BASE_SMA", "BASE_EMA"]:
        # Build the indicator from price (as in baseline), combine with ML ‘entry permission’
        if strat_name == "BASE_SMA":
            rule = (close > close.rolling(20, min_periods=1).mean()).astype(int)
        else:
            rule = (close > close.ewm(span=20, adjust=False).mean()).astype(int)

        # Final trading signal = ML_signal AND rule
        ml = sig.reindex(close.index).fillna(0).astype(int)
        final = (ml & rule).astype(int)
        # Reuse metric calc
        ret = compute_returns(close)
        strat_ret = ret.shift(-1).fillna(0.0) * final
        total = (1.0 + strat_ret).prod() - 1.0
        trades = int(final.diff().abs().fillna(0).sum())
        hit_rate = float((strat_ret > 0).mean())

        metrics.append({
            "type": "keyword",
            "strategy": strat_name,        # trading rule used
            "model": model,                # learning model
            "data": data_type,             # BASE vs EXT features
            "asset": asset,
            "freq": freq,
            "total_return": total,
            "trades": trades,
            "hit_rate": hit_rate,
        })

    out = pd.DataFrame(metrics)
    # Write per-asset file (append pattern)
    dst = DRIVE_ARTEFACTS / asset / f"metrics_keywords_{freq}.csv"
    if dst.exists():
        prev = pd.read_csv(dst)
        out = pd.concat([prev, out], ignore_index=True)
        out = out.drop_duplicates(subset=["strategy", "model", "data", "asset", "freq"], keep="last")
    write_csv(out, dst)

    # Accumulate for root rollups
    if freq == "D":
        root_kw_D.append(out.assign(type="keyword"))
    else:
        root_kw_W.append(out.assign(type="keyword"))

    # Optional copy of signal files into repo-friendly tree (so the app can read them there too)
    # Determine repo signal destination
    dst_sig = REPO_ARTEFACTS / asset / fp.name
    dst_sig.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(fp, dst_sig)
    print(f"[ok] KW metrics -> {asset} {fp.name}")

# ---- Root-level rollups (leaderboards + keyword summaries)
def build_leaderboard(asset: str, freq: str):
    # Prefer keyword metrics if present; otherwise baseline
    kw_fp = DRIVE_ARTEFACTS / asset / f"metrics_keywords_{freq}.csv"
    base_fp = DRIVE_ARTEFACTS / asset / f"metrics_baseline_{freq}.csv"
    if kw_fp.exists():
        df_kw = pd.read_csv(kw_fp)
        # Sort by total_return desc
        board = df_kw.sort_values("total_return", ascending=False).head(10)
    else:
        board = pd.read_csv(base_fp).sort_values("total_return", ascending=False)
    write_csv(board, DRIVE_ARTEFACTS / asset / f"leaderboard_{freq}.csv")

for a in TICKER_MAP:
    build_leaderboard(a, "D")
    print(f"[ok] leaderboard -> {a} D")
    build_leaderboard(a, "W")
    print(f"[ok] leaderboard -> {a} W")

root_D = pd.concat(root_kw_D, ignore_index=True) if root_kw_D else pd.DataFrame(columns=[
    "type","strategy","model","data","asset","freq","total_return","trades","hit_rate"
])
root_W = pd.concat(root_kw_W, ignore_index=True) if root_kw_W else pd.DataFrame(columns=[
    "type","strategy","model","data","asset","freq","total_return","trades","hit_rate"
])

# Baseline summary for reference
base_rows_D, base_rows_W = [], []
for a in TICKER_MAP:
    base_rows_D.append(pd.read_csv(DRIVE_ARTEFACTS / a / "metrics_baseline_D.csv").assign(asset=a, freq="D"))
    base_rows_W.append(pd.read_csv(DRIVE_ARTEFACTS / a / "metrics_baseline_W.csv").assign(asset=a, freq="W"))
base_D = pd.concat(base_rows_D, ignore_index=True)
base_W = pd.concat(base_rows_W, ignore_index=True)

# Root summary CSVs expected by the app (include both baseline + keyword if you want):
# Here we export keywords-only summaries at root for the app's landing stats,
# and keep per-asset baseline files for the UI tables.
write_csv(base_D.groupby("asset").apply(lambda x: x.sort_values("total_return", ascending=False).head(1)).reset_index(drop=True),
          DRIVE_ARTEFACTS / "metrics_summary_D.csv")
write_csv(base_W.groupby("asset").apply(lambda x: x.sort_values("total_return", ascending=False).head(1)).reset_index(drop=True),
          DRIVE_ARTEFACTS / "metrics_summary_W.csv")

write_csv(root_D, DRIVE_ARTEFACTS / "metrics_keywords_D.csv")
write_csv(root_W, DRIVE_ARTEFACTS / "metrics_keywords_W.csv")

# ---- Mirror Drive artefacts -> Repo artefacts (so Block 2 can push)
def mirror(src: Path, dst: Path):
    dst.mkdir(parents=True, exist_ok=True)
    for sroot, _, files in os.walk(src):
        rel = os.path.relpath(sroot, src)
        droot = dst / rel
        Path(droot).mkdir(parents=True, exist_ok=True)
        for fn in files:
            sp = Path(sroot) / fn
            dp = Path(droot) / fn
            shutil.copy2(sp, dp)

mirror(DRIVE_ARTEFACTS, REPO_ARTEFACTS)
print("[sync] Drive artefacts -> Repo artefacts")

# ---- Optional quick check (sizes + row counts for keyword metric files)
for f in sorted(glob.glob(str(REPO_ARTEFACTS / "*/*/metrics_keywords_*.csv"))):
    try:
        rows = len(pd.read_csv(f))
    except Exception:
        rows = -1
    print(f"{f}  size={os.stat(f).st_size}  rows={rows}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mounted at /content/drive
Input: /content/drive/MyDrive/gt-markets/data/processed/merged_financial_trends_engineered_2025-09-07.csv
Drive artefacts: /content/drive/MyDrive/gt-markets/AppDemo/artefacts
Repo artefacts: /content/gt-markets/AppDemo/artefacts
Merged shape: (2609, 254)
Sample cols: ['BTC-USD Close', 'CL=F Close', 'DXY Close', 'GC=F Close', 'USDCNY=X Close', 'BTC-USD Open', 'CL=F Open', 'DXY Open', 'GC=F Open', 'USDCNY=X Open', 'BTC-USD High', 'CL=F High']
[ok] BTC D: baseline rows=2
[ok] BTC W: baseline rows=2
[ok] GOLD D: baseline rows=2
[ok] GOLD W: baseline rows=2
[ok] OIL D: baseline rows=2
[ok] OIL W: baseline rows=2
[ok] USDCNY D: baseline rows=2
[ok] USDCNY W: baseline rows=2
Summary rows: 8
[ok] KW metrics -> BTC signals_KW_GRU_w30_BASE_D.csv
[ok] KW metrics -> BTC signals_KW_GRU_w30_BASE_W.csv
[ok] KW metrics -> BTC signals_KW_GRU_w30_EXT_

/tmp/ipython-input-3387736147.py:308: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  write_csv(base_D.groupby("asset").apply(lambda x: x.sort_values("total_return", ascending=False).head(1)).reset_index(drop=True),
/tmp/ipython-input-3387736147.py:310: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  write_csv(base_W.groupby("asset").apply(lambda x: x.sort_values("total_return", ascending=False).head(1)).reset_

[sync] Drive artefacts -> Repo artefacts


In [14]:
# =========================
# BLOCK 2 — Commit & push AppDemo/artefacts only (Colab-safe)
# =========================

from pathlib import Path
import subprocess as sp
import os, re, getpass, datetime, shutil, sys

OWNER = "brendonhuynhbp-hub"
REPO  = "gt-markets"
CLONE_DIR = Path("/content/gt-markets")
ARTEFACTS_REL = Path("AppDemo/artefacts")   # only this path will be staged/committed/pushed

def sh(cmd, cwd=None, check=False, capture=False):
    """Run a shell command. If capture=True, returns CompletedProcess with stdout/stderr."""
    if capture:
        return sp.run(cmd, cwd=cwd, check=check, text=True, capture_output=True)
    return sp.run(cmd, cwd=cwd, check=check, text=True)

def parse_owner_repo(url: str):
    m = re.match(r"https?://github\.com/([^/]+)/([^/\.]+)(?:\.git)?/?$", url)
    if m: return m.group(1), m.group(2)
    m = re.match(r"git@github\.com:([^/]+)/([^/\.]+)(?:\.git)?$", url)
    if m: return m.group(1), m.group(2)
    raise ValueError(f"Unrecognised origin URL: {url}")

# --- 1) Ensure a clean repo folder (auto-clone if missing or not a git repo)
if CLONE_DIR.exists() and not (CLONE_DIR / ".git").exists():
    print(f"⚠️ Found existing {CLONE_DIR} without .git/ — removing it so we can clone clean.")
    shutil.rmtree(CLONE_DIR)

if not (CLONE_DIR / ".git").exists():
    print("Repo not found at /content/gt-markets. Attempting clone…")
    CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)

    try_pub = sh(["git", "clone", f"https://github.com/{OWNER}/{REPO}.git", str(CLONE_DIR)], capture=True)
    if try_pub.returncode != 0:
        print("Public clone failed (likely private repo). Will use PAT.")
        token = getpass.getpass("GitHub Personal Access Token (PAT) [scope: repo]: ")
        pat_clone_url = f"https://{token}@github.com/{OWNER}/{REPO}.git"
        try_priv = sh(["git", "clone", pat_clone_url, str(CLONE_DIR)], capture=True)
        if try_priv.returncode != 0:
            print(try_priv.stdout)
            print(try_priv.stderr)
            raise RuntimeError("Clone with PAT failed. Check PAT scopes (repo) and SSO access.")
        print("✅ Cloned with PAT.")
        del token
    else:
        print("✅ Cloned without auth (public).")
else:
    print("Repo already present at /content/gt-markets.")

os.chdir(CLONE_DIR)
print(f"Repo root: {CLONE_DIR}")

# --- 2) Confirm remote + branch
origin = sh(["git", "remote", "get-url", "origin"], capture=True)
if origin.returncode != 0:
    raise RuntimeError("No 'origin' remote set. The clone looks broken.")
origin_url = origin.stdout.strip()
print(f"Remote: {origin_url}")

branch_res = sh(["git", "rev-parse", "--abbrev-ref", "HEAD"], capture=True)
if branch_res.returncode != 0:
    raise RuntimeError("Cannot determine current branch.")
branch = branch_res.stdout.strip()
print(f"Branch: {branch}")

try:
    owner_detected, repo_detected = parse_owner_repo(origin_url)
except Exception:
    owner_detected, repo_detected = OWNER, REPO

# --- 3) Git identity for this repo
sh(["git", "config", "user.name", "brendonhuynhbp-hub"], check=True)
sh(["git", "config", "user.email", "brendonhuynh.bp@gmail.com"], check=True)

# --- 4) Pull latest (fetch + rebase) to avoid non–fast-forward rejects
print("\nFetching and rebasing on latest remote…")
fetch = sh(["git", "fetch", "origin"], capture=True)
if fetch.returncode != 0:
    print(fetch.stderr)
    raise RuntimeError("git fetch failed.")

pull = sh(["git", "pull", "--rebase", "origin", branch], capture=True)
if pull.returncode != 0:
    print(pull.stdout)
    print(pull.stderr)
    raise RuntimeError("git pull --rebase failed. Resolve conflicts then re-run.")
else:
    print(pull.stdout.strip() or "Up to date.")

# --- 5) Stage ONLY the artefacts directory
artefacts_abs = CLONE_DIR / ARTEFACTS_REL
if not artefacts_abs.exists():
    raise FileNotFoundError(f"Expected artefacts at {artefacts_abs}. Run Block 1 first to write files here.")

print(f"\nStaging: {ARTEFACTS_REL}")
add = sh(["git", "add", "-A", str(ARTEFACTS_REL)], capture=True)
if add.returncode != 0:
    print(add.stderr)
    raise RuntimeError("git add failed.")

# If nothing staged, finish quietly
diff_cached = sh(["git", "diff", "--cached", "--name-only"], capture=True)
staged_list = [l for l in diff_cached.stdout.strip().splitlines() if l.strip()]
if not staged_list:
    print("No changes to commit in AppDemo/artefacts.")
    status = sh(["git", "status", "--short"], capture=True)
    print(status.stdout)
    sys.exit(0)

# --- 6) Commit
stamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
msg = f"AppDemo: update artefacts ({stamp})"
commit = sh(["git", "commit", "-m", msg], capture=True)
if commit.returncode != 0:
    print(commit.stdout)
    print(commit.stderr)
    raise RuntimeError("git commit failed.")
print(commit.stdout.strip())

# --- 7) Push with one-off PAT URL (does not change origin remote)
print("\nGitHub Personal Access Token (PAT) will be used for this push only.")
print("Required scopes: repo (and SSO must be authorised if your org enforces it).")
token = getpass.getpass("GitHub Personal Access Token (PAT): ")

push_url = f"https://{token}@github.com/{owner_detected}/{repo_detected}.git"
print(f"Pushing to https://github.com/{owner_detected}/{repo_detected} on '{branch}' …")
env = os.environ.copy()
env["GIT_ASKPASS"] = "echo"  # suppress interactive prompts

push = sp.run(["git", "push", push_url, branch], text=True, capture_output=True, env=env, cwd=CLONE_DIR)
if push.returncode != 0:
    print(push.stdout)
    print(push.stderr)
    raise RuntimeError("git push failed. If remote changed again, re-run: fetch, pull --rebase, then push.")
else:
    print("✅ Push complete.")
    del token

# --- 8) Final status
print("\nOn branch", branch)
status = sh(["git", "status", "--short"], capture=True)
print(status.stdout or "Working tree clean.")


Repo already present at /content/gt-markets.
Repo root: /content/gt-markets
Remote: https://github.com/brendonhuynhbp-hub/gt-markets.git
Branch: main

Fetching and rebasing on latest remote…

error: cannot pull with rebase: You have unstaged changes.
error: please commit or stash them.



RuntimeError: git pull --rebase failed. Resolve conflicts then re-run.